# 새 기상 입력과 공식 취약구간 비교

온도 8모델과 결빙 30모델의 **완료 결과**를 읽고, 추가 입력의 효과와 공식 취약구간 비교를 검토합니다. 실행 모델의 교체나 새 독립 평가를 뜻하지 않습니다. 전체 학습은 `config/reanalysis_*_20261031.json`과 `scripts/evaluate_reanalysis_temperature.py`, `scripts/evaluate_reanalysis_icing.py`의 지역 제외 계획을 따릅니다. 상세 방법은 `docs/research/REANALYSIS_INPUTS_20261031.md`와 `OFFICIAL_CHAINAGE_COMPARISON_20261031.md`를 참고하세요.

실행 순서: 검산 기록 확인 → 온도 비교 → 결빙 비교 → 위치 보간의 오차 → 동률과 연속 구간 확인. 원본 작업 공간에서 위에서 아래로 실행합니다. 외부 API를 새로 호출하지 않습니다.

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'config/reanalysis_icing_20261031.json').exists())
P=ROOT/'outputs/private'
def read(path): return json.loads(path.read_text(encoding='utf-8'))
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
for name in ['reanalysis_temperature_20261031_v1','reanalysis_icing_20261031_v1']:
    audit=read(P/name/'independent_verification.json')
    assert audit['status']=='passed' and audit['summary_sha256']==sha(P/name/'summary.json')
    assert not audit['protected_outcomes_read'] and not audit['new_four_site_outcomes_read']
assert read(P/'vulnerability_map_comparison_20261031_v1/independent_verification.json')['status']=='passed'
print('완료된 연구와 지도 검산 기록 확인')

## 같은 평가 자료의 온도 오차

동일한 310,091유효쌍을 비교합니다. 추가 입력과 기존 모델은 같은 지역 분리와 학습 설정을 사용했습니다. ERA5-Land는 지연된 재분석으로, 실시간 도로 노면 관측이나 500m 공간 해상도를 제공하지 않습니다.

In [ ]:
temperature=read(P/'reanalysis_temperature_20261031_v1/summary.json')
rows=[]
for name, report in temperature['reports'].items():
    pooled=report['regression']['pooled']
    rows.append({'후보':name, **pooled, '채택 조건 통과':report['eligible']})
display(pd.DataFrame(rows))
assert not any(r['eligible'] for r in temperature['reports'].values())

## 지역을 제외한 결빙 판별과 사건 손실

허용 23곳을 활용하되 이미 검토한 9개 개발 지점에서 반복 비교합니다. 새 독립 성능으로 해석하지 않습니다. 지역 내부에서 임계값과 후보를 선택하고 ASOS 기온 기준선 및 동일 학습 대조와 비교합니다. 기존에 탐지한 사건의 손실도 함께 확인합니다.

In [ ]:
icing=read(P/'reanalysis_icing_20261031_v1/summary.json')
names=['asos_zero','baseline_mean_f1','selected_mean_f1','selected_worst_winter_f1']
display(pd.DataFrame([{'후보':name,'일별 평균 F1':icing['reports'][name]['all']['f1_fixed_population'], **icing['reports'][name]['events']} for name in names]))
display(pd.Series(icing['gates'],name='채택 조건 통과'))
assert not icing['primary_success']
print('기존 모델 유지. 7일 블록 민감도:',icing['sensitivity'])

## 암거 이정 보간과 별도 위치 진단

동일 시설을 양방향에 연결한 908개 기록은 서로 다른 908개 측량점이 아닙니다. 서로 다른 위치는 454곳입니다. 시설 자료의 공간 제외 오차와 별도 CCTV 진단을 구별해 확인합니다. 최신 목록의 위치는 보간한 추정치이며 공식 측량으로 검증한 도로 이정이 아닙니다.

In [ ]:
alignment=read(P/'official_chainage_alignment_20261031_v1/summary.json')
display(pd.DataFrame(alignment['validation']).T)
display(pd.Series(alignment['cctv_diagnostic'],name='별도 CCTV 진단'))
inventory=pd.read_csv(P/'vulnerability_map_comparison_20261031_v1/inventory_mapping.csv')
display(inventory.loc[inventory.candidate_segments.gt(0),['version','inventory_id','route_nm','headquarters','start_km','end_km','candidate_segments']])
assert not alignment['official_chainage_verified']

## 동률을 포함한 노출과 연속 확인 구간

길이 가중 75백분위 이상에 동률을 모두 포함하므로 선정 길이가 정확히 25%가 되지 않습니다. 318개 보고 구간은 방향별 8개 연속 현장 확인 구간으로 묶었습니다. 실제 Ice 확률이나 공식 A/B/C 등급으로 해석하지 않습니다. 기상자료를 공유하는 구간을 독립 발견으로 세지 않습니다.

In [ ]:
viewer=P/'official_inventory_viewer_20261031_v1'
display(pd.read_csv(viewer/'threshold_ties.csv'))
display(pd.read_csv(viewer/'corridors.csv'))
assert len(pd.read_csv(viewer/'corridors.csv'))==8
print(viewer/'index.html')